# 04 - Pose Inference
Run the trained pose model on all videos to extract keypoint coordinates.

In [ ]:
# ===== CONFIGURATION =====
# GitHub -- do not change
GITHUB_REPO_URL = "https://github.com/kaarthik-balakrishnan/LightningPoseTrack.git"
GIT_BRANCH = "main"

# Google Drive root -- change this to match your Drive structure
DRIVE_ROOT = "/content/drive/My Drive/PigBehavior"  # <-- SET THIS to your root

# Derived paths (change if your folders are at custom locations)
DRIVE_RAW_VIDEOS = f"{DRIVE_ROOT}/raw_videos"               # Input: session video folders
DRIVE_MODELS = f"{DRIVE_ROOT}/trained_models"         # Input: trained model checkpoints
DRIVE_POSE_OUTPUTS = f"{DRIVE_ROOT}/pose_outputs"     # Output: pose prediction CSVs

CONFIDENCE_THRESHOLD = 0.5

# Per-camera pixel bounds (corners of the visible arena in order:
# top-left, top-right, bottom-right, bottom-left).
# Any keypoint predicted outside these bounds is discarded.
PIXEL_POINTS = {
    1: [(153, 388), (1415, 388), (1415, 907), (153, 907)],
    2: [(252, 464), (1754, 432), (1761, 640), (249, 640)],
    3: [(166, 421), (1537, 453), (1535, 610), (160, 608)],
    4: [(360, 360), (1348, 384), (1368, 840), (367, 835)],
}

# Google Drive folder ID (for reference)
DRIVE_FOLDER_ID = "1X_41ZW3HfwVeft2lPld3XNqXsdxRDIwb"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, sys
REPO_DIR = "/content/LightningPoseTrack"
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

In [ ]:
# Install Lightning Pose + inference dependencies
!pip install --quiet "lightning-pose[all]" opencv-python pandas numpy pyarrow imageio[ffmpeg]

In [ ]:
from pathlib import Path

model_dir = Path(DRIVE_MODELS) / "pose_model"
if model_dir.exists() and (model_dir / "config.yaml").exists():
    print(f"Model directory found: {model_dir}")
    print(f"  Contents: {[p.name for p in model_dir.iterdir()]}")
else:
    print(f"Model directory not found at {model_dir}. Complete notebook 03 first.")

In [ ]:
from src.io.video_inventory import scan_videos, parse_camera_from_filename

df = scan_videos(DRIVE_RAW_VIDEOS)
print(f"Found {len(df)} videos to process")
df[["filename", "session", "camera", "frame_count", "duration_min"]]

In [ ]:
import cv2
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

from lightning_pose.api import Model

# Precompute per-camera bounding boxes from PIXEL_POINTS
CAMERA_BOUNDS = {}
for cam, corners in PIXEL_POINTS.items():
    xs = [p[0] for p in corners]
    ys = [p[1] for p in corners]
    CAMERA_BOUNDS[cam] = (min(xs), max(xs), min(ys), max(ys))
print("Camera bounds (x_min, x_max, y_min, y_max):")
for cam, (xmn, xmx, ymn, ymx) in sorted(CAMERA_BOUNDS.items()):
    print(f"  Camera {cam}: x=[{xmn}, {xmx}]  y=[{ymn}, {ymx}]")

def filter_out_of_bounds(kps, confs, cam):
    """Set keypoints outside the camera's pixel bounds to NaN."""
    if cam not in CAMERA_BOUNDS:
        return kps, confs
    xmin, xmax, ymin, ymax = CAMERA_BOUNDS[cam]
    n_kp = len(kps) // 2
    for i in range(n_kp):
        x, y = kps[i*2], kps[i*2+1]
        if x < xmin or x > xmax or y < ymin or y > ymax:
            kps[i*2] = float("nan")
            kps[i*2+1] = float("nan")
            confs[i] = 0.0
    return kps, confs

# Load the trained model from the model directory (contains config.yaml + .ckpt)
model_dir = Path(DRIVE_MODELS) / "pose_model"
model = Model.from_dir(str(model_dir))
print(f"Model loaded from {model_dir}")

pose_output_dir = Path(DRIVE_POSE_OUTPUTS)
pose_output_dir.mkdir(parents=True, exist_ok=True)

KEYPOINT_NAMES = [
    "snout", "left_ear", "right_ear", "neck",
    "shoulders", "mid_back", "hip", "tail_base",
]

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing videos"):
    video_path = Path(DRIVE_RAW_VIDEOS) / row["path"]
    session = row["session"]
    camera = row["camera"]
    stem = Path(row["filename"]).stem

    out_file = pose_output_dir / session / f"{stem}_cam{camera}_pose.parquet"
    if out_file.exists():
        print(f"Skipping {out_file.name} (already exists)")
        continue

    # Use OpenCV to read frames (DALI cannot handle .asf)
    cap = cv2.VideoCapture(str(video_path))
    kp_list, conf_list = [], []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = model.predict_frame(frame_rgb)
        kps = result["keypoints"].ravel()
        confs = result["confidence"]
        kps, confs = filter_out_of_bounds(kps, confs, camera)
        kp_list.append(kps)
        conf_list.append(confs)
    cap.release()

    # Build DataFrame matching the same column schema as predict_on_video_file output
    n_kp = len(KEYPOINT_NAMES)
    gen_df = pd.DataFrame({
        "frame": range(len(kp_list)),
        **{f"{kp}_x": [k[i*2] for k in kp_list] for i, kp in enumerate(KEYPOINT_NAMES)},
        **{f"{kp}_y": [k[i*2+1] for k in kp_list] for i, kp in enumerate(KEYPOINT_NAMES)},
        **{f"{kp}_likelihood": [c[i] for c in conf_list] for i, kp in enumerate(KEYPOINT_NAMES)},
    })
    out_file.parent.mkdir(parents=True, exist_ok=True)
    gen_df.to_parquet(str(out_file))
    print(f"Saved: {out_file} ({len(gen_df)} frames)")

print("\nPose inference complete!")


In [ ]:
# Verify outputs
output_files = list(pose_output_dir.rglob("*.parquet"))
print(f"Total pose output files: {len(output_files)}")
for f in output_files:
    print(f"  {f.relative_to(pose_output_dir)}")